# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-rewritten-rejected/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-rewritten-rejected/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-rewritten-rejected/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-rewritten-rejected/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                               \
                       base             base_inv                      ft   
0           .Today (0.0239)      urrenc (0.0217)         .Today (0.0322)   
1          .Second (0.0114)       pos (5.16e-03)      Buccane (9.22e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)      .Second (8.12e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)        /Area (6.74e-03)   
4            .au (4.88e-03)      gons (3.33e-03)    /entities (4.36e-03)   
5      /problems (4.03e-03)        �� (2.01e-03)         fter (4.21e-03)   
6            aru (3.91e-03)      ejec (1.95e-03)    /operator (3.97e-03)   
7      /entities (2.96e-03)      azon (1.95e-03)     /problem (3.85e-03)   
8       /problem (2.69e-03)        دي (1.95e-03)    /problems (3.60e-03)   
9          /bind (2.69e-03)     fácil (1.79e-03)          aru (3.39e-03)   
10      /respond (2.61e-03)      anth (1.79e-03)    /activity (3.19e-03)   
11         /Math (2.61e-03)     posix (1.73e-03)        /Math (3.19e-03)   
12          fter (2.46e-03)  essional (1.63e-03)     /respond (3.08e-03)   
13    confidence (2.30e-03)  Optional (1.53e-03)          .au (2.99e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)          eft (2.40e-03)   
15     /activity (2.17e-03)    Phones (1.43e-03)       soever (2.18e-03)   
16   persistence (2.17e-03)        vs (1.39e-03)         oire (2.12e-03)   
17     belonging (2.03e-03)       med (1.27e-03)   confidence (2.00e-03)   
18          ilot (1.97e-03)      orst (1.27e-03)          ERM (2.00e-03)   
19     .AddRange (1.85e-03)  >Welcome (1.23e-03)        /bind (1.93e-03)   

                                                                       \
                 ft_inv                  diff                diff_inv   
0       urrenc (0.0156)          mek (0.0342)            ans (0.0229)   
1        pos (8.36e-03)       OURNAL (0.0221)            ali (0.0216)   
2        act (6.96e-03)     souvenir (0.0104)            olo (0.0127)   
3     askell (3.83e-03)  /customer (6.74e-03)          uil (9.58e-03)   
4       gons (2.72e-03)       dane (5.25e-03)          éri (8.18e-03)   
5      fácil (2.72e-03)      lator (5.25e-03)      passing (7.02e-03)   
6       ejec (2.12e-03)        IFF (4.94e-03)          ern (6.38e-03)   
7         �� (2.06e-03)         gf (4.09e-03)          ter (5.83e-03)   
8   essional (1.87e-03)      -sama (3.85e-03)            ü (4.52e-03)   
9     Phones (1.70e-03)    oppable (3.19e-03)            з (4.24e-03)   
10      anth (1.70e-03)      legen (3.19e-03)           th (4.12e-03)   
11         次 (1.70e-03)         }? (2.99e-03)           ét (3.65e-03)   
12      azon (1.65e-03)       beth (2.99e-03)          Fac (3.22e-03)   
13      Vers (1.60e-03)      SHALL (2.64e-03)           uy (3.22e-03)   
14        vs (1.50e-03)      ursos (2.64e-03)   absolutely (3.11e-03)   
15       div (1.50e-03)       hebt (2.64e-03)         ders (2.93e-03)   
16       dbl (1.33e-03)       прав (2.49e-03)         aina (2.84e-03)   
17        دي (1.33e-03)        kit (2.33e-03)      Content (2.67e-03)   
18       775 (1.24e-03)        ama (2.33e-03)         Peel (2.67e-03)   
19  Optional (1.24e-03)    ###\n\n (2.33e-03)            省 (2.67e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9375)   
1           The (0.0452)      contador (0.1309)          The (0.0221)   
2           Let (0.0156)           메 (8.36e-03)          Let (0.0152)   
3            In (0.0146)         иск (3.49e-03)         In (9.83e-03)   
4         ### (4.21e-03)     Produto (2.12e-03)        ### (3.83e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (2.47e-03)   
6         For (1.28e-03)      Resets (1.11e-05)        For (1.33e-03)   
7          As (9.69e-04)     Detalle (8.64e-06)          1 (7.55e-04)   
8         

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0417)   
1        /entities (0.0272)       (us (5.07e-03)       /entities (0.0277)   
2        /problems (0.0176)      sagt (4.46e-03)       /problems (0.0190)   
3         .Today (8.85e-03)       ]]; (3.94e-03)        .Today (9.28e-03)   
4        /global (8.61e-03)        że (3.48e-03)       /global (9.03e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (7.02e-03)   
6           /job (6.68e-03)      -ves (2.70e-03)          /job (6.59e-03)   
7   /preferences (6.07e-03)     zeigt (2.70e-03)       /layout (5.83e-03)   
8        /layout (5.71e-03)       ($. (2.70e-03)  /preferences (5.83e-03)   
9      /provider (5.04e-03)       ')" (2.70e-03)     /provider (5.46e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)       /crypto (4.97e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)   /connection (4.24e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)          /reg (4.24e-03)   
13  /environment (4.06e-03)   kontrol (1.98e-03)       /engine (3.88e-03)   
14       /engine (3.94e-03)    spiele (1.98e-03)      /logging (3.88e-03)   
15      /logging (3.94e-03)       (!! (1.98e-03)    WHATSOEVER (3.88e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)      /effects (3.75e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)  /environment (3.65e-03)   
18       /dialog (3.36e-03)       )": (1.45e-03)       /dialog (3.52e-03)   
19      /effects (3.36e-03)       fas (1.45e-03)       /entity (3.42e-03)   

                                                                  \
                 ft_inv                  diff           diff_inv   
0        .vn (8.00e-03)         anno (0.0205)      allo (0.0170)   
1        (us (4.85e-03)      Schwarz (0.0106)   Passage (0.0170)   
2       sagt (4.27e-03)        Regel (0.0103)        ís (0.0132)   
3        ]]; (4.27e-03)      weise (5.00e-03)      be (8.54e-03)   
4         że (3.13e-03)      props (4.70e-03)    assi (7.54e-03)   
5       -ves (2.94e-03)       umer (4.70e-03)   resco (7.54e-03)   
6      zeigt (2.44e-03)    initely (4.43e-03)    adel (5.19e-03)   
7        ($. (2.44e-03)       rite (3.78e-03)    umar (5.19e-03)   
8     testim (2.44e-03)   /comment (3.45e-03)   ulton (5.19e-03)   
9        ')" (2.29e-03)    -switch (2.94e-03)    fres (4.85e-03)   
10     spons (2.15e-03)       Rena (2.76e-03)    ende (4.85e-03)   
11     feliz (2.15e-03)       cine (2.69e-03)    erre (4.58e-03)   
12     lesen (2.15e-03)  /customer (2.59e-03)      ét (4.03e-03)   
13    spiele (2.03e-03)        ovo (2.59e-03)   ighth (4.03e-03)   
14       (!! (2.03e-03)   /respond (2.52e-03)    plat (3.56e-03)   
15   kontrol (1.90e-03)      agnar (2.44e-03)       省 (3.14e-03)   
16     scrut (1.57e-03)       Sink (2.44e-03)     ssl (2.94e-03)   
17      helf (1.48e-03)        :]. (2.29e-03)    rend (2.94e-03)   
18       )": (1.48e-03)        Kod (2.23e-03)    shaw (2.44e-03)   
19       fas (1.48e-03)     ocrine (2.15e-03)    gest (2.44e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5898)     contador (0.8555)        , (0.6289)   
1       and (0.1465)    kontrol (7.39e-03)      and (0.1279)   
2       the (0.1289)         bö (5.77e-03)      the (0.1128)   
3        in (0.0559)         �� (5.77e-03)       in (0.0554)   
4       ' ' (0.0479)   karakter (5.77e-03)      ' ' (0.0417)   
5         a (0.0131)         �� (4.49e-03)        a (0.0123)   
6       ( (3.28e-03)      subur (3.49e-03)      ( (3.89e-03)   
7      to (2.94e-03)     testim (2.72e-03)     to (2.87e-03)   
8      of (2.75e-03)      KANJI (2.72e-03)     of (2.61e-03)   
9      by (2.32e-03)       samt (2.72e-03)     by (2.44e-03)   
10    for (1.70e-03)     kosten (2.12e-03)    for (1.87e-03)

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0196)       /entities (0.0272)   
1         /problem (0.0140)   /Register (0.0113)        /problem (0.0144)   
2      /problems (9.19e-03)    testim (7.01e-03)     /problems (9.81e-03)   
3        /global (6.69e-03)      sagt (6.58e-03)       /global (6.80e-03)   
4   /environment (6.00e-03)     asign (5.31e-03)   /connection (6.07e-03)   
5      /provider (5.87e-03)       -ie (4.92e-03)     /provider (6.06e-03)   
6    /connection (5.74e-03)     zeigt (4.03e-03)  /environment (5.85e-03)   
7         .Today (5.74e-03)        że (3.99e-03)     /customer (5.77e-03)   
8        /manage (5.64e-03)      -ves (3.29e-03)        .Today (5.69e-03)   
9      /customer (4.72e-03)         ť (2.94e-03)       /manage (5.12e-03)   
10  /preferences (4.25e-03)   personn (2.81e-03)  /preferences (3.98e-03)   
11       /dialog (3.37e-03)     probs (2.78e-03)       /dialog (3.40e-03)   
12       /shared (3.34e-03)      elig (2.58e-03)       /shared (3.38e-03)   
13      /account (3.22e-03)    ):\n\n (2.36e-03)      /account (3.33e-03)   
14       /entity (3.19e-03)      roku (2.35e-03)      libertin (3.29e-03)   
15      libertin (3.11e-03)     lesen (2.30e-03)       /entity (3.18e-03)   
16       /layout (2.92e-03)  ,,,,,,,, (2.23e-03)       /layout (3.00e-03)   
17          .Try (2.82e-03)       )": (2.21e-03)      /effects (2.90e-03)   
18      /effects (2.73e-03)    spiele (2.12e-03)          .Try (2.77e-03)   
19        /legal (2.64e-03)       esl (2.09e-03)       /crypto (2.69e-03)   

                                                                    \
                 ft_inv                  diff             diff_inv   
0          .vn (0.0202)       umer (9.85e-03)        ís (8.22e-03)   
1    /Register (0.0108)  reachable (9.26e-03)      beat (5.16e-03)   
2       sagt (6.36e-03)       anus (6.82e-03)        he (4.96e-03)   
3     testim (6.22e-03)       indo (6.15e-03)      Beat (4.81e-03)   
4        -ie (5.07e-03)      Regel (5.51e-03)         省 (4.05e-03)   
5      asign (4.84e-03)     Trails (4.31e-03)         装 (3.77e-03)   
6         że (3.97e-03)        ovo (4.29e-03)      allo (3.54e-03)   
7      zeigt (3.96e-03)       anno (3.44e-03)   Passage (3.06e-03)   
8       -ves (3.54e-03)  /customer (2.91e-03)         � (2.93e-03)   
9          ť (3.27e-03)      agnar (2.78e-03)       bru (2.69e-03)   
10     probs (2.79e-03)       acea (2.41e-03)     tutto (2.61e-03)   
11   personn (2.68e-03)      weise (2.24e-03)       esa (2.52e-03)   
12      elig (2.62e-03)      inite (2.17e-03)         х (2.51e-03)   
13      roku (2.40e-03)     prises (2.14e-03)      band (2.51e-03)   
14    ):\n\n (2.31e-03)     sovere (2.09e-03)      pass (2.41e-03)   
15       )": (2.27e-03)    eniable (2.02e-03)        be (2.36e-03)   
16    spiele (2.23e-03)         rg (1.99e-03)     issan (2.12e-03)   
17     lesen (2.20e-03)      antee (1.99e-03)      hele (1.91e-03)   
18  ,,,,,,,, (2.12e-03)      indiv (1.95e-03)      Humb (1.88e-03)   
19       (us (2.05e-03)     ocrine (1.89e-03)       Fac (1.56e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8033)     contador (0.9622)         , (0.8178)   
1        ' ' (0.1076)    kontrol (3.12e-03)       ' ' (0.0967)   
2        the (0.0411)   karakter (2.50e-03)       the (0.0385)   
3        and (0.0300)       rekl (1.59e-03)       and (0.0296)   
4       in (5.95e-03)         bö (1.38e-03)      in (5.53e-03)   
5        ( (4.36e-03)       zoek (1.14e-03)       ( (5.05e-03)   
6       's (2.96e-03)    Produto (9.67e-04)      's (2.59e-03)   
7        a (1.68e-03)     testim (9.64e-04)       a (1.56e-03)   
8       to (9.64e-04)     bilder (8.72e-04)      to (9.28e-04)   
9        . (6.66e-04)         �� (7.63e-04)       . (6.81e

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                 \
                       base                        ft                 diff   
0           .Today (0.0258)           .Today (0.0299)          << (0.0187)   
1        .Second (9.70e-03)            aru (6.22e-03)          GA (0.0141)   
2        Buccane (6.73e-03)        .Second (5.54e-03)     red (6.94e-03) ✅   
3        /Area (5.76e-03) ✅        /Area (5.37e-03) ✅     zeros (6.67e-03)   
4            .au (5.29e-03)           fter (5.37e-03)        dr (5.86e-03)   
5            aru (5.09e-03)        Buccane (5.26e-03)      this (5.81e-03)   
6    /problems (3.21e-03) ✅            .au (3.47e-03)         0 (5.58e-03)   
7     confidence (2.83e-03)   confidence (2.97e-03) ✅        is (5.05e-03)   
8           fter (2.72e-03)           ilot (2.79e-03)     seems (4.76e-03)   
9        /Math (2.50e-03) ✅    /entities (2.73e-03) ✅   sport (4.39e-03) ✅   
10   /entities (2.48e-03) ✅        /Math (2.73e-03) ✅         p (4.36e-03)   
11          ilot (2.43e-03)     /problem (2.46e-03) ✅       cur (4.32e-03)   
12    /problem (2.19e-03) ✅    /operator (2.46e-03) ✅         � (4.28e-03)   
13       /bind (2.19e-03) ✅    /activity (2.43e-03) ✅   appears (3.68e-03)   
14   /activity (2.08e-03) ✅    /problems (2.19e-03) ✅       bip (3.34e-03)   
15   persistence (1.97e-03)            eft (1.73e-03)   asics (3.08e-03) ✅   
16    /respond (1.93e-03) ✅   /community (1.71e-03) ✅         R (2.95e-03)   
17   /operator (1.91e-03) ✅    belonging (1.71e-03) ✅     '   ' (2.93e-03)   
18     belonging (1.87e-03)           [sub (1.67e-03)        -p (2.77e-03)   
19   .AddRange (1.60e-03) ✅     /respond (1.57e-03) ✅         1 (2.75e-03)   

                layer_14                                                  \
                    base                    ft                      diff   
0            To (0.7266)           To (0.9414)             else (0.0131)   
1           ### (0.1260)          The (0.0195)               10 (0.0109)   
2            ** (0.0767)        Let (0.0172) ✅              elo (0.0103)   
3         Let (0.0527) ✅         In (8.12e-03)           eral (9.64e-03)   
4           The (0.0118)        ### (4.94e-03)            Cod (7.51e-03)   
5   Certainly (1.40e-03)          A (1.82e-03)          chein (5.49e-03)   
6        Sure (1.09e-03)        For (1.04e-03)            999 (5.16e-03)   
7          ## (7.51e-04)         ** (7.74e-04)              5 (4.27e-03)   
8          In (5.15e-04)        1 (6.68e-04) ✅          eldon (4.27e-03)   
9     Given (2.24e-04) ✅         As (5.53e-04)           reau (3.13e-03)   
10    First (2.02e-04) ✅       We (4.31e-04) ✅          earch (2.93e-03)   
11    Alright (1.23e-04)  Certainly (4.06e-04)              ( (2.93e-03)   
12       We (1.08e-04) ✅          I (3.81e-04)      working (2.93e-03) ✅   
13          1 (1.08e-04)    First (3.05e-04) ✅   practicing (2.59e-03) ✅   
14       #### (9.54e-05)         It (2.88e-04)          wrong (2.59e-03)   
15        ``` (9.54e-05)       Sure (2.71e-04)             no (2.43e-03)   
16     Here (8.96e-05) ✅     This (2.17e-04) ✅      writing (2.15e-03) ✅   
17     This (8.27e-05) ✅          H (1.35e-04)              或 (2.15e-03)   
18         As (5.21e-05)          S (9.54e-05)           rito (2.15e-03)   
19        For (5.10e-05)    Given (9.06e-05) ✅          agnar (2.01e-03)   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.4141)           To (0.5742)               no (0.0197)  
1            ** (0.2852)          ### (0.2119)                ( (0.0102)  
2           ### (0.2520)           ** (0.1650)            ([[ (9.89e-03)  
3         Let (0.0234) ✅        Let (0.0325) ✅           agen (8.97e-03)  
4           The (0.0161)        The (8.18e-03)            elo (7.20e-03)  
5   Certainly (2.47e-03)  Certainly (2.08e-03)            for (5.98e-03)  
6        Sure (1.50e-03)      

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 's (0.0430)                's (0.0401)   
1                 -> (0.0318)        /problem (0.0301) ✅   
2                the (0.0277)       /entities (0.0205) ✅   
3         /problem (0.0263) ✅       /problems (0.0205) ✅   
4            solve (0.0253) ✅           solve (0.0152) ✅   
5          problem (0.0197) ✅         /manage (0.0120) ✅   
6              :\n\n (0.0193)                 , (0.0103)   
7        /entities (0.0177) ✅             the (9.12e-03)   
8        /problems (0.0145) ✅    understand (6.21e-03) ✅   
9                you (0.0125)          .Today (5.01e-03)   
10       /manage (8.80e-03) ✅             you (4.19e-03)   
11          '\n\n' (7.68e-03)  /preferences (3.96e-03) ✅   
12            this (6.77e-03)       /global (3.68e-03) ✅   
13            your (5.93e-03)       /crypto (3.36e-03) ✅   
14             :\n (5.73e-03)              ’s (3.25e-03)   
15            '\n' (4.20e-03)     /provider (3.25e-03) ✅   
16        .Today (4.06e-03) ✅           seems (2.94e-03)   
17    understand (3.97e-03) ✅       analyze (2.69e-03) ✅   
18       address (3.91e-03) ✅        solves (2.63e-03) ✅   
19              ’s (3.66e-03)   /connection (2.27e-03) ✅   
20  /preferences (3.55e-03) ✅      /effects (2.24e-03) ✅   
21       /global (3.52e-03) ✅       /object (2.20e-03) ✅   
22        solves (3.49e-03) ✅              we (2.17e-03)   
23       analyze (2.50e-03) ✅          /job (2.16e-03) ✅   
24     /provider (2.43e-03) ✅        tackle (2.11e-03) ✅   
25       /crypto (2.41e-03) ✅       address (1.94e-03) ✅   
26          /job (2.29e-03) ✅       /layout (1.82e-03) ✅   
27      problems (2.10e-03) ✅     /activity (1.73e-03) ✅   
28        puzzle (2.10e-03) ✅         break (1.67e-03) ✅   
29           seems (2.04e-03)  /application (1.51e-03) ✅   
30               , (1.90e-03)       /shared (1.47e-03) ✅   
31       /layout (1.88e-03) ✅           :\n\n (1.41e-03)   
32      /logging (1.85e-03) ✅       /engine (1.41e-03) ✅   
33       /object (1.84e-03) ✅              is (1.36e-03)   
34   /connection (1.70e-03) ✅             /pr (1.29e-03)   
35          math (1.68e-03) ✅      /logging (1.24e-03) ✅   
36      question (1.55e-03) ✅      /respond (1.17e-03) ✅   
37     /activity (1.51e-03) ✅  /controllers (1.09e-03) ✅   
38     statement (1.43e-03) ✅            your (1.09e-03)   
39               : (1.41e-03)       /entity (1.00e-03) ✅   
40       /engine (1.33e-03) ✅          .Round (9.66e-04)   
41          step (1.32e-03) ✅        /legal (9.56e-04) ✅   
42       puzzles (1.29e-03) ✅            /con (9.49e-04)   
43              we (1.22e-03)             /pl (9.26e-04)   
44        tackle (1.21e-03) ✅   understands (7.86e-04) ✅   
45  /environment (1.19e-03) ✅        decode (7.43e-04) ✅   
46      /effects (1.18e-03) ✅         appears (7.39e-04)   
47       /shared (1.11e-03) ✅          /reg (6.70e-04) ✅   
48          begins (1.10e-03)  /environment (6.56e-04) ✅   
49  /application (1.03e-03) ✅     /customer (6.48e-04) ✅   
50       /entity (1.01e-03) ✅      /testing (6.17e-04) ✅   
51        /legal (9.90e-04) ✅        /tasks (5.31e-04) ✅   
52          word (9.05e-04) ✅      /vendors (5.27e-04) ✅   
53             for (8.06e-04)        /event (4.76e-04) ✅   
54        solved (7.57e-04) ✅      /company (4.54e-04) ✅   
55        answer (7.21e-04) ✅            /man (4.15e-04)   
56              is (7.05e-04)                              
57          task (6.92e-04) ✅                              
58        decode (6.47e-04) ✅                              
59      /testing (6.45e-04) ✅                              
60          /reg (5.93e-04) ✅                              
61           /pr (5.87e-04) ✅                              
62          .Round (5.70e-04)                              
63        /tasks (5.27e-04) ✅                              
64      WHATSOEVER (5.15e-04)                              
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                               \
                 pos_-3                pos_-1                  pos_0   
0        inite (0.0320)          mek (0.0342)        ocrine (0.0238)   
1         rupt (0.0234)       OURNAL (0.0221)        rite (8.73e-03)   
2          cur (0.0161)     souvenir (0.0104)     Verdana (8.24e-03)   
3          ixa (0.0142)  /customer (6.74e-03)         tog (7.93e-03)   
4          間 (7.87e-03)       dane (5.25e-03)        umer (4.67e-03)   
5       visa (7.63e-03)      lator (5.25e-03)       onica (3.88e-03)   
6     rupted (7.63e-03)        IFF (4.94e-03)  characters (3.11e-03)   
7    Masters (6.53e-03)         gf (4.09e-03)       props (2.84e-03)   
8      igned (5.77e-03)      -sama (3.85e-03)         ild (2.75e-03)   
9        ail (5.58e-03)    oppable (3.19e-03)       weise (2.50e-03)   
10     Kirby (4.91e-03)      legen (3.19e-03)        ynos (2.35e-03)   
11       ibi (3.97e-03)         }? (2.99e-03)        /reg (2.21e-03)   
12       Koh (3.28e-03)       beth (2.99e-03)         eed (2.21e-03)   
13      oods (3.08e-03)      SHALL (2.64e-03)        inte (2.14e-03)   
14    phases (2.99e-03)      ursos (2.64e-03)        ickt (2.08e-03)   
15       RTS (2.90e-03)       hebt (2.64e-03)    -Regular (2.01e-03)   
16    encial (2.81e-03)       прав (2.49e-03)       .exec (1.95e-03)   
17    Member (2.81e-03)        kit (2.33e-03)      kommen (1.89e-03)   
18      LEEP (2.81e-03)        ama (2.33e-03)        cate (1.62e-03)   
19       hdl (2.64e-03)    ###\n\n (2.33e-03)     _script (1.62e-03)   

                                                                      \
                   pos_1                 pos_2                 pos_3   
0        ocrine (0.0143)         anno (0.0275)         anno (0.0226)   
1     Verdana (8.73e-03)      Regel (8.36e-03)        Regel (0.0103)   
2       Regel (6.16e-03)      props (7.87e-03)    initely (4.30e-03)   
3       props (3.10e-03)     ocrine (7.87e-03)    Schwarz (4.06e-03)   
4       leine (3.01e-03)      sumer (6.53e-03)      props (3.69e-03)   
5        umer (2.91e-03)       Sink (6.32e-03)       Sink (3.69e-03)   
6   /question (2.82e-03)       umer (4.91e-03)      sumer (3.57e-03)   
7        Sink (2.49e-03)   Validity (4.49e-03)    -switch (3.46e-03)   
8           � (2.41e-03)    initely (3.71e-03)       Rena (3.16e-03)   
9    /comment (2.33e-03)          � (3.49e-03)   /comment (3.16e-03)   
10      .jdbc (2.33e-03)      utive (2.72e-03)       umer (3.16e-03)   
11       anno (2.33e-03)   /comment (2.72e-03)          � (2.96e-03)   
12   Validity (2.33e-03)       cine (2.55e-03)      inite (2.78e-03)   
13    nonzero (2.20e-03)     ovable (2.40e-03)   Validity (2.78e-03)   
14       avar (2.00e-03)  reachable (2.18e-03)     ocrine (2.53e-03)   
15      utive (2.00e-03)        Kod (2.12e-03)      spiel (2.53e-03)   
16   /respond (2.00e-03)      salle (1.98e-03)  /customer (2.30e-03)   
17        tog (1.94e-03)      spiel (1.92e-03)        tos (2.11e-03)   
18     iguous (1.88e-03)       rite (1.92e-03)    /detail (1.85e-03)   
19      onica (1.82e-03)      inite (1.92e-03)       Born (1.85e-03)   

                                                                      \
                  pos_10                pos_50               pos_100   
0          umer (0.0186)         umer (0.0115)       anus (9.70e-03)   
1         Regel (0.0144)  reachable (8.91e-03)       indo (9.09e-03)   
2   /customer (8.24e-03)       anus (6.96e-03)       umer (8.54e-03)   
3        anno (7.26e-03)     Trails (5.77e-03)  reachable (8.54e-03)   
4     Schwarz (5.65e-03)       indo (5.40e-03)     Trails (4.73e-03)   
5   reachable (4.27e-03)        ovo (5.25e-03)        ovo (4.18e-03)   
6     iflower (3.77e-03)      Regel (3.72e-03)       acea (4.03e-03)   
7         Kod (3.02e-03)  /customer (3.28e-03)      Regel (3.80e-03)   
8       agnar (2.93e-03)      agnar (3.08e-03)        Méd (2.87e-03)   
9       phase (2.93e-03)       anno (2.64e-03)      we

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                                \
                 pos_-3               pos_-1                    pos_0   
0           -> (0.3018)          << (0.0187)          ocrine (0.0188)   
1           -> (0.0648)          GA (0.0141)            rite (0.0112)   
2            1 (0.0382)     red (6.94e-03) ✅     Verdana (8.71e-03) ✅   
3           <- (0.0297)     zeros (6.67e-03)           tog (7.39e-03)   
4            A (0.0175)        dr (5.86e-03)          umer (4.91e-03)   
5       Blue (0.0164) ✅      this (5.81e-03)         onica (3.71e-03)   
6            0 (0.0114)         0 (5.58e-03)           ild (3.24e-03)   
7          d (9.87e-03)        is (5.05e-03)  characters (3.01e-03) ✅   
8          - (7.39e-03)     seems (4.76e-03)         weise (2.69e-03)   
9          J (6.97e-03)   sport (4.39e-03) ✅       props (2.66e-03) ✅   
10        In (6.95e-03)         p (4.36e-03)        kommen (2.32e-03)   
11      Only (6.92e-03)       cur (4.32e-03)    -Regular (2.21e-03) ✅   
12      ->\n (5.67e-03)         � (4.28e-03)           eed (2.16e-03)   
13    blue (5.00e-03) ✅   appears (3.68e-03)          ickt (2.11e-03)   
14       And (4.66e-03)       bip (3.34e-03)       .exec (2.09e-03) ✅   
15   Green (4.39e-03) ✅   asics (3.08e-03) ✅          /reg (1.97e-03)   
16         D (3.69e-03)         R (2.95e-03)          ynos (1.90e-03)   
17     Red (3.16e-03) ✅     '   ' (2.93e-03)           FAT (1.90e-03)   
18  orange (3.10e-03) ✅        -p (2.77e-03)          inte (1.81e-03)   
19   green (2.75e-03) ✅         1 (2.75e-03)     _script (1.72e-03) ✅   

                                                                              \
                     pos_1                    pos_2                    pos_3   
0          ocrine (0.0137)               , (0.0143)       witness (0.0239) ✅   
1       Verdana (8.28e-03)               " (0.0130)             our (0.0179)   
2         Regel (5.88e-03)       witness (0.0130) ✅               S (0.0130)   
3          umer (3.45e-03)             our (0.0110)   witnesses (9.09e-03) ✅   
4       props (3.35e-03) ✅        dear (6.10e-03) ✅            my (7.83e-03)   
5         leine (2.86e-03)             � (6.01e-03)           ' ' (7.61e-03)   
6   /question (2.86e-03) ✅           ' ' (5.70e-03)        dear (6.53e-03) ✅   
7          Sink (2.83e-03)   witnesses (5.67e-03) ✅          mine (6.25e-03)   
8     nonzero (2.48e-03) ✅            sg (5.54e-03)            ![ (6.23e-03)   
9    Validity (2.38e-03) ✅             ! (5.38e-03)          ours (5.86e-03)   
10            � (2.38e-03)            ![ (5.08e-03)          mine (5.22e-03)   
11        .jdbc (2.16e-03)        anyone (4.70e-03)            rg (5.06e-03)   
12         avar (2.14e-03)            my (4.37e-03)             # (5.06e-03)   
13         anno (2.14e-03)             # (4.27e-03)            Ts (4.25e-03)   
14   /comment (2.10e-03) ✅             S (4.04e-03)           Ary (4.16e-03)   
15        utive (2.03e-03)           amt (3.45e-03)             , (3.98e-03)   
16       iguous (1.97e-03)           rev (3.01e-03)           pat (3.41e-03)   
17   /respond (1.91e-03) ✅        poor (2.75e-03) ✅         Sir (3.33e-03) ✅   
18        onica (1.85e-03)        soul (2.73e-03) ✅    proposed (3.05e-03) ✅   
19          tos (1.85e-03)          cade (2.63e-03)             " (2.96e-03)   

                    layer_14                            \
                      pos_-3                    pos_-1   
0           Extra (0.0199) ✅             else (0.0131)   
1            Jarvis (0.0152)               10 (0.0109)   
2           ocket (7.30e-03)              elo (0.0103)   
3          arrera (6.32e-03)           eral (9.64e-03)   
4        -extra (5.14e-03) ✅            Cod (7.51e-03)   
5            иров (5.03e-03)          chein (5.49e-03)   
6         extra (4.44e-03) ✅            999 (5.16e-03)   
7             eth (3.18e-03)              5 (4.27e-03)   
8             Nom (2.92e-03)          eldon (4.27e-03)   
9         